# Notebook 00: Database Setup & Full Fitting Pipeline

**Run this notebook once before any analysis notebooks.**  
It is idempotent — safe to re-run; existing rows are skipped unless you set `FORCE_REFIT = True`.

Steps:
1. Initialize the SQLite database
2. Ingest SPARC rotation curves and galaxy metadata
3. **Seed Rational Taper** results from the pre-computed CSV (avoids expensive re-fitting)
4. Fit NFW, Fixed MOND, and Free MOND for all 171 galaxies
5. Report convergence summary

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path so `src` is importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.database import (
    Galaxy, ModelFit, delete,
    get_session, init_db, insert_model_fit, query_fits_as_dataframe,
)
from src.ingest import ingest_massmodels_mrt
from src.fit import run_fits_for_galaxy
from src.utils import get_project_root, setup_logger

logger = setup_logger('notebook_00')
project_root = get_project_root()

# Set to True to delete existing fits and refit from scratch
FORCE_REFIT = False

print(f'Project root: {project_root}')

Project root: C:\Projects\ISM\tapered-model-comparison


## Step 1: Initialize Database

In [2]:
engine = init_db()
session = get_session(engine)
print('Database initialized.')

2026-03-09 09:48:58 | INFO     | src.database | Database initialized at C:\Projects\ISM\tapered-model-comparison\data\processed\galaxy_dynamics.db


Database initialized.


## Step 2: Ingest SPARC Data

Parses `MassModels_Lelli2016c.mrt` (rotation curves) and links galaxy metadata from `SPARC_Lelli2016c.mrt`.

In [3]:
mrt_path = project_root / 'data' / 'raw' / 'MassModels_Lelli2016c.mrt'
meta_path = project_root / 'data' / 'raw' / 'SPARC_Lelli2016c.mrt'

assert mrt_path.exists(), f'MRT file not found: {mrt_path}'
assert meta_path.exists(), f'Metadata file not found: {meta_path}'

ingested_ids = ingest_massmodels_mrt(
    massmodels_path=str(mrt_path),
    metadata_path=str(meta_path),
)
print(f'Ingested {len(ingested_ids)} galaxies.')

2026-03-09 09:48:58 | INFO     | src.ingest | Parsed MRT file: 3391 data points across 175 galaxies
2026-03-09 09:48:58 | INFO     | src.ingest | Parsed metadata for 175 galaxies
2026-03-09 09:48:58 | INFO     | src.database | Database initialized at C:\Projects\ISM\tapered-model-comparison\data\processed\galaxy_dynamics.db
2026-03-09 09:48:58 | INFO     | src.database | Upserted galaxy: CamB
2026-03-09 09:48:58 | INFO     | src.database | Inserted 9 radial profiles for CamB
2026-03-09 09:48:58 | INFO     | src.database | Upserted galaxy: D512-2
2026-03-09 09:48:58 | INFO     | src.database | Inserted 4 radial profiles for D512-2
2026-03-09 09:48:58 | INFO     | src.database | Upserted galaxy: D564-8
2026-03-09 09:48:58 | INFO     | src.database | Inserted 6 radial profiles for D564-8
2026-03-09 09:48:58 | INFO     | src.database | Upserted galaxy: D631-7
2026-03-09 09:48:58 | INFO     | src.database | Inserted 16 radial profiles for D631-7
2026-03-09 09:48:58 | INFO     | src.database

Ingested 175 galaxies.


In [4]:
# Verify galaxy count
from sqlalchemy import func
from src.database import RadialProfile

n_gal = session.query(func.count(Galaxy.galaxy_id)).scalar()
n_prof = session.query(func.count(RadialProfile.profile_id)).scalar()
print(f'Galaxies in DB:          {n_gal}')
print(f'Total radial profile rows: {n_prof}')
assert n_gal >= 171, f'Expected at least 171 galaxies, found {n_gal}'

Galaxies in DB:          175
Total radial profile rows: 3391


## Step 3: Seed Rational Taper Results from CSV

The Rational Taper fits were computed in the sibling project (`baryonic-omega-analysis`) and saved to  
`data/raw/Schneider_2026_SPARC_Fit_Parameters.csv`. We seed these directly to avoid expensive re-fitting.

Column mapping:
- `Tapered_omega` → `param1`
- `Tapered_Rt` → `param2`
- `Tapered_BIC` → `bic`
- `Tapered_chi2r * N_pts` → `chi_squared` (approximate reconstruction)
- `method_version` = `"v1_seeded"`

In [5]:
csv_path = project_root / 'data' / 'raw' / 'Schneider_2026_SPARC_Fit_Parameters.csv'
assert csv_path.exists(), f'CSV not found: {csv_path}'

csv_df = pd.read_csv(csv_path)
print(f'CSV rows: {len(csv_df)}')
print(csv_df[['GalaxyID', 'N_pts', 'Tapered_omega', 'Tapered_Rt', 'Tapered_BIC', 'Tapered_converged']].head())

CSV rows: 175
  GalaxyID  N_pts  Tapered_omega  Tapered_Rt  Tapered_BIC  Tapered_converged
0     CamB      9       1.113497    8.950000    45.323908               True
1   D512-2      4      20.785488    1.427526     2.970875               True
2   D564-8      6       8.022627    6.467550     4.282406               True
3   D631-7     16       6.801220   35.950000    49.978488               True
4   DDO064     14      25.713959    1.506292     9.059213               True


In [6]:
# Check which galaxies already have a rational_taper fit
existing_rt = query_fits_as_dataframe(session, model_name='rational_taper')
already_seeded = set(existing_rt['galaxy_id'].values) if not existing_rt.empty else set()
print(f'Already seeded: {len(already_seeded)} Rational Taper fits')

seeded = 0
skipped = 0
for _, row in csv_df.iterrows():
    gid = str(row['GalaxyID'])

    # Skip if not in DB (not in our 171-galaxy clean sample)
    gal = session.get(Galaxy, gid)
    if gal is None:
        continue

    if not FORCE_REFIT and gid in already_seeded:
        skipped += 1
        continue

    if FORCE_REFIT and gid in already_seeded:
        session.execute(
            delete(ModelFit)
            .where(ModelFit.galaxy_id == gid)
            .where(ModelFit.model_name == 'rational_taper')
        )
        session.flush()

    n_pts = int(row['N_pts'])
    chi2r = float(row['Tapered_chi2r']) if pd.notna(row['Tapered_chi2r']) else None
    chi2 = float(chi2r * n_pts) if chi2r is not None else None
    reduced_chi2 = chi2r

    fit_dict = {
        'galaxy_id':          gid,
        'model_name':         'rational_taper',
        'n_params':           2,
        'chi_squared':        chi2,
        'reduced_chi_squared': reduced_chi2,
        'bic':                float(row['Tapered_BIC']) if pd.notna(row['Tapered_BIC']) else None,
        'residuals_rmse':     float(row['Tapered_RMSE']) if pd.notna(row['Tapered_RMSE']) else None,
        'n_points':           n_pts,
        'converged':          bool(row['Tapered_converged']),
        'flag_v_obs_lt_v_bary': bool(row['Flag_Vobs_lt_Vbary']) if pd.notna(row['Flag_Vobs_lt_Vbary']) else False,
        'method_version':     'v1_seeded',
        'upsilon_disk':       0.5,
        'upsilon_bulge':      0.7,
        'param1':             float(row['Tapered_omega']) if pd.notna(row['Tapered_omega']) else None,
        'param1_err':         float(row['Tapered_omega_err']) if pd.notna(row['Tapered_omega_err']) else None,
        'param2':             float(row['Tapered_Rt']) if pd.notna(row['Tapered_Rt']) else None,
        'param2_err':         float(row['Tapered_Rt_err']) if pd.notna(row['Tapered_Rt_err']) else None,
    }
    insert_model_fit(session, fit_dict)
    seeded += 1

print(f'Seeded: {seeded} Rational Taper fits')
print(f'Skipped (already present): {skipped}')

Already seeded: 175 Rational Taper fits
Seeded: 0 Rational Taper fits
Skipped (already present): 175


## Step 4: Fit NFW, Fixed MOND, and Free MOND

The Rational Taper is already seeded above and will be skipped by the `run_fits_for_galaxy` skip-check.  
This step only fits the three remaining models.

In [7]:
galaxy_ids = [row[0] for row in session.query(Galaxy.galaxy_id).all()]
print(f'Fitting {len(galaxy_ids)} galaxies for NFW, MOND fixed, MOND free...')

try:
    from tqdm.notebook import tqdm
    galaxy_iter = tqdm(galaxy_ids, desc='Fitting', unit='galaxy')
except ImportError:
    galaxy_iter = galaxy_ids

converge_counts = {'nfw': 0, 'mond_fixed': 0, 'mond_free': 0}
models_to_fit = ['nfw', 'mond_fixed', 'mond_free']

for i, gid in enumerate(galaxy_iter):
    if not hasattr(galaxy_iter, 'set_description') and i % 20 == 0:
        print(f'  Progress: {i}/{len(galaxy_ids)}')
    try:
        results = run_fits_for_galaxy(
            galaxy_id=gid,
            session=session,
            models=models_to_fit,
            force=FORCE_REFIT,
        )
        for m, r in results.items():
            if r.converged:
                converge_counts[m] = converge_counts.get(m, 0) + 1
    except Exception as exc:
        print(f'  ERROR {gid}: {exc}')

print('\n=== Convergence Summary ===')
for model, cnt in converge_counts.items():
    print(f'  {model:20s}: {cnt}/{len(galaxy_ids)} converged ({100*cnt/len(galaxy_ids):.1f}%)')

Fitting 175 galaxies for NFW, MOND fixed, MOND free...


Fitting:   0%|          | 0/175 [00:00<?, ?galaxy/s]

2026-03-09 09:48:59 | INFO     | src.fit | CamB [nfw]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | CamB [mond_fixed]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | CamB [mond_free]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | D512-2 [nfw]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | D512-2 [mond_fixed]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | D512-2 [mond_free]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | D564-8 [nfw]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | D564-8 [mond_fixed]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | INFO     | src.fit | D564-8 [mond_free]: already fitted, skipping (use --force to refit)
2026-03-09 09:48:59 | 


=== Convergence Summary ===
  nfw                 : 0/175 converged (0.0%)
  mond_fixed          : 0/175 converged (0.0%)
  mond_free           : 0/175 converged (0.0%)


## Step 5: Final Database Check

In [8]:
fits_all = query_fits_as_dataframe(session)
print(f'Total model_fits rows: {len(fits_all)}')

summary = fits_all.groupby('model_name').agg(
    n_rows=('galaxy_id', 'count'),
    n_converged=('converged', 'sum'),
    median_bic=('bic', 'median'),
).reset_index()
summary['pct_converged'] = (summary['n_converged'] / summary['n_rows'] * 100).round(1)
print('\nPer-model summary:')
print(summary.to_string(index=False))

Total model_fits rows: 700

Per-model summary:
    model_name  n_rows  n_converged  median_bic  pct_converged
    mond_fixed     175          175  169.904111          100.0
     mond_free     175          175   48.514641          100.0
           nfw     175          175   23.038485          100.0
rational_taper     175          175   18.916244          100.0


## Setup Complete

The database is populated and all four models are fitted. Proceed to **Notebook 01** for sample validation.